# Feeana: Dual-Head Fine-Tuning + ONNX Export (Google Colab T4)

Minimal, reproducible notebook targeting Google Colab T4 GPU runtime.

### Overview:
- **Base Model**: configurable — set once in the *Config* cell below (`bert-base-multilingual-cased` for mBERT, or the DistilXLM-R default). Public models, no HF token required.
- **Architecture**: Shared encoder + 15-way `issue` head + 3-way `polarity` head
- **Training**: Full fine-tuning of the shared encoder
- **Export**: ONNX FP32 → INT8 quantization + smoke test (Steps 9–10)
- **Dependencies**: Minimal installation targeting training dependencies without touching preinstalled Colab numpy/pandas

### Config: Select Base Model

**This is the ONE line you change** to switch which model is fine-tuned and exported.

- `bert-base-multilingual-cased` → mBERT (output tag: `mbert`)
- `nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large` → DistilXLM-R (default, tag: `distilxlmr`)

All downstream steps (training, diagnostics, ONNX export, smoke test) read this env var automatically.

In [ ]:
import os
os.environ["FEEANA_MODEL_NAME"] = ""
print(f"[CONFIG] Base model: {os.environ['FEEANA_MODEL_NAME']}")

### Step 1: Verify CUDA & GPU Environment

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not enabled! Go to Runtime > Change runtime type > Select T4 GPU.")

!nvidia-smi

### Step 2: Install Minimal Required Dependencies

Upgrades `torchao` to `>=0.16.0` while avoiding unnecessary `-U` upgrades or version locks on preinstalled Colab packages (`numpy`, `pandas`, `torch`).

In [ ]:
!pip install -q \
  "torchao>=0.16.0" \
  "transformers>=4.38.0" \
  "datasets>=2.18.0" \
  "evaluate>=0.4.0" \
  "accelerate>=0.27.0" \
  "scikit-learn>=1.3.0" \
  "onnx>=1.15.0" \
  "onnxruntime>=1.17.0"

### Step 3: Verify Core Package Imports & Versions

In [ ]:
import sys
import subprocess

cmd = [
    sys.executable, "-c",
    "import torch, torchao, transformers; " \
    "print('torch:       ', torch.__version__); " \
    "print('torchao:     ', torchao.__version__); " \
    "print('transformers:', transformers.__version__)"
]
res = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(res.stdout)
print("[PASS] Core imports verified without version conflicts!")

### Step 4: Smoke Test — Dual-Head Model Instantiation & Forward Pass (with AMP/FP16)

Verifies that `DualHeadModel` instantiates and executes a forward pass under AMP without FP16/float32 dtype mismatches.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "scripts/training")

import torch
from finetune import DualHeadModel, MODEL_NAME, NUM_ISSUES, NUM_POLARITIES

print(f"[SMOKE TEST] Loading base model '{MODEL_NAME}'...")
model = DualHeadModel(MODEL_NAME, NUM_ISSUES, NUM_POLARITIES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

dummy_ids = torch.zeros((2, 16), dtype=torch.long, device=device)
dummy_mask = torch.ones((2, 16), dtype=torch.long, device=device)

use_amp = (device.type == "cuda")
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, enabled=use_amp):
        out = model(dummy_ids, dummy_mask)

assert out["issue_logits"].shape == (2, NUM_ISSUES), f"Issue logits shape mismatch: {out['issue_logits'].shape}"
assert out["polarity_logits"].shape == (2, NUM_POLARITIES), f"Polarity logits shape mismatch: {out['polarity_logits'].shape}"

print(f"\n[PASS] DualHeadModel successfully initialized and verified under AMP on {device}!")
print(f"  - Issue logits shape:    {out['issue_logits'].shape}")
print(f"  - Polarity logits shape: {out['polarity_logits'].shape}")

### Step 4: Upload and verify repository training files

Upload the complete local `scripts/training/` directory into Colab so it appears at `/content/scripts/training/`. Include the Python scripts used below and the `data/` directory.

In [ ]:
from pathlib import Path
import pandas as pd

required_files = [
    Path("scripts/training/finetune.py"),
    Path("scripts/training/checkpoint_paths.py"),
    Path("scripts/training/export_model_onnx.py"),
    Path("scripts/training/evaluate_test.py"),
    Path("scripts/training/compare_distilxlmr_runtimes.py"),
    Path("scripts/training/conditional_val_metrics.py"),
    Path("scripts/training/data/train.csv"),
    Path("scripts/training/data/val.csv"),
    Path("scripts/training/data/test.csv"),
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

for path in required_files:
    if path.suffix == ".csv":
        print(f"[FOUND] {path} ({len(pd.read_csv(path)):,} rows)")
    else:
        print(f"[FOUND] {path}")
print("[READY] Training files verified.")

### Step 6: Execute Phase 2 Fine-Tuning Run

The base model comes from the *Config* cell (`FEEANA_MODEL_NAME`), so no `--model-name` flag is needed here. Batch 16 fits the T4 for the DistilXLM-R base; if you hit a CUDA OOM with mBERT (~110M params), do NOT lower `--batch-size` to 8.

In [ ]:
!python scripts/training/finetune.py --epochs 8 --patience 8 --batch-size 16 --lr 2e-5 --seed 42

### Step 7: Verify Checkpoint Artifacts & Saved Model

In [ ]:
import sys
from pathlib import Path
import json

sys.path.insert(0, "scripts/training")
from finetune import MODEL_NAME, resolve_tag

tag = resolve_tag(MODEL_NAME)
ckpt_file = Path("scripts/training/checkpoints") / tag / "best_model.pt"
json_file = Path("scripts/training/checkpoints") / tag / "label_mappings.json"

if ckpt_file.exists() and json_file.exists():
    size_mb = ckpt_file.stat().st_size / (1024 * 1024)
    print(f"[SUCCESS] Checkpoint saved: {ckpt_file} ({size_mb:.2f} MB)")
    with open(json_file, encoding="utf-8") as f:
        mappings = json.load(f)
    print(f"[SUCCESS] Label mappings saved: {json_file}")
    print(f"  - Issue labels ({mappings['issue']['num_labels']}): {list(mappings['issue']['id2label'].values())[:3]}...")
    print(f"  - Polarity labels ({mappings['polarity']['num_labels']}): {list(mappings['polarity']['id2label'].values())}")
    print("\nTraining completed successfully! Download scripts/training/checkpoints/ for Phase 3/4.")
else:
    print("WARNING: Checkpoints not found. Please review training log above.")

### Step 8: Export FP32 and INT8 ONNX

Exports the trained checkpoint to FP32 ONNX, retains the FP32 intermediate, and creates the required INT8 post-training quantized artifact.

In [ ]:
!python scripts/training/export_model_onnx.py \
    --model-name nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large \
    --out-dir scripts/training/exports \
    --keep-fp32

### Step 9: Verify exported artifacts

Checks that the FP32 and INT8 files, tokenizer files, configuration, and label mappings were created.

In [ ]:
from pathlib import Path

export_dir = Path("scripts/training/exports/distilxlmr")
required_outputs = [
    "int8-fp32.onnx",
    "int8.onnx",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "config.json",
    "label_mappings.json",
]
missing = [name for name in required_outputs if not (export_dir / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing exported files: {missing}")

print(f"{'File':<32} {'Size (MB)':>12}")
print("-" * 46)
for name in required_outputs:
    path = export_dir / name
    print(f"{name:<32} {path.stat().st_size / (1024 * 1024):>12.2f}")
print("[PASS] FP32, INT8, tokenizer, configuration, and label mappings verified.")

### Step 10: Conditional validation diagnostics

Runs the focused validation report for named issues, `uncategorized` detection, and polarity conditioned on issue correctness.

In [ ]:
!python scripts/training/conditional_val_metrics.py \
    --checkpoint scripts/training/checkpoints/distilxlmr/best_model.pt \
    --data scripts/training/data/val.csv \
    --output scripts/training/reports/conditional_val_metrics.json

### Step 11: Held-out test evaluation

Evaluates the newly trained PyTorch checkpoint on `scripts/training/data/test.csv`.

In [ ]:
!python scripts/training/evaluate_test.py

### Step 12: Runtime parity

Compares PyTorch FP32, FP32 ONNX, and INT8 ONNX using the same checkpoint and `scripts/training/data/test.csv`.

In [ ]:
!python scripts/training/compare_distilxlmr_runtimes.py \
    --checkpoint scripts/training/checkpoints/distilxlmr/best_model.pt \
    --fp32-onnx scripts/training/exports/distilxlmr/int8-fp32.onnx \
    --int8-onnx scripts/training/exports/distilxlmr/int8.onnx \
    --data scripts/training/data/test.csv \
    --output scripts/training/reports/distilxlmr_runtime_parity.json

### Step 13: Package isolated outputs

Creates a curated archive under `feeana-distilxlmr-output/`. Extract it outside the repository so existing files are not overwritten.

In [ ]:
from pathlib import Path
import shutil

output_root = Path("feeana-distilxlmr-output")
if output_root.exists():
    shutil.rmtree(output_root)

checkpoint_output = output_root / "checkpoints" / "distilxlmr"
export_output = output_root / "exports" / "distilxlmr"
report_output = output_root / "reports"
checkpoint_output.mkdir(parents=True)
export_output.mkdir(parents=True)
report_output.mkdir(parents=True)

shutil.copytree(
    "scripts/training/checkpoints/distilxlmr",
    checkpoint_output,
    dirs_exist_ok=True,
)
shutil.copytree(
    "scripts/training/exports/distilxlmr",
    export_output,
    dirs_exist_ok=True,
)

required_reports = [
    "conditional_val_metrics.json",
    "test_evaluation_report_distilxlmr.json",
    "distilxlmr_runtime_parity.json",
]
for report_name in required_reports:
    source = Path("scripts/training/reports") / report_name
    if source.exists():
        shutil.copy2(source, report_output / report_name)

training_reports = sorted(
    Path("scripts/training/reports").glob("distilxlmr_training_run_*.json")
)
if not training_reports:
    raise FileNotFoundError("No DistilXLM-R training report was found.")
shutil.copy2(training_reports[-1], report_output / training_reports[-1].name)

archive = shutil.make_archive(
    "feeana-distilxlmr-output",
    "zip",
    root_dir=".",
    base_dir=output_root.name,
)
print(f"[READY] Created {archive}")

from google.colab import files
files.download(archive)